In [1]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd

In [2]:
sys.path.append("..")  # esto sube un nivel desde Scripts_visual_block
# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "block"
subj = "sub-V1001"

# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")

mne.utils.set_config('SUBJECTS_DIR', r'\\wsl$\Ubuntu-20.04\usr\local\freesurfer\subjects', set_env=True)

import os, re, warnings



📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_block
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\ICA_block
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_block
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_block\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_block\fwd
inverse_pat

# LECTURA EPOCHS

In [3]:
# Construimos lista subj_epochs_block a partir de archivos .fif
subj_epochs_block = []
pattern = re.compile(r"^(.*?)_epochs_")

for fname in os.listdir(epochs_clean_path):
    if fname.endswith("-epo.fif"):
        match = pattern.match(fname)
        if match:
            subj_epochs_block.append(match.group(1))

subj_epochs_block = list(set(subj_epochs_block))

print("\nSujetos encontrados en epochs_clean_path:")
print(subj_epochs_block)


Sujetos encontrados en epochs_clean_path:
['sub-V1100', 'sub-V1057', 'sub-V1011', 'sub-V1113', 'sub-V1013', 'sub-V1032', 'sub-V1007', 'sub-V1028', 'sub-V1092', 'sub-V1027', 'sub-V1073', 'sub-V1074', 'sub-V1102', 'sub-V1115', 'sub-V1036', 'sub-V1049', 'sub-V1024', 'sub-V1106', 'sub-V1050', 'sub-V1003', 'sub-V1093', 'sub-V1042', 'sub-V1088', 'sub-V1084', 'sub-V1019', 'sub-V1094', 'sub-V1098', 'sub-V1038', 'sub-V1031', 'sub-V1087', 'sub-V1114', 'sub-V1048', 'sub-V1062', 'sub-V1099', 'sub-V1085', 'sub-V1080', 'sub-V1009', 'sub-V1055', 'sub-V1079', 'sub-V1078', 'sub-V1095', 'sub-V1004', 'sub-V1016', 'sub-V1097', 'sub-V1053', 'sub-V1070', 'sub-V1040', 'sub-V1101', 'sub-V1012', 'sub-V1076', 'sub-V1025', 'sub-V1081', 'sub-V1008', 'sub-V1026', 'sub-V1111', 'sub-V1117', 'sub-V1020', 'sub-V1035', 'sub-V1045', 'sub-V1005', 'sub-V1015', 'sub-V1107', 'sub-V1065', 'sub-V1052', 'sub-V1086', 'sub-V1064', 'sub-V1022', 'sub-V1002', 'sub-V1039', 'sub-V1066', 'sub-V1104', 'sub-V1116', 'sub-V1068', 'sub-V1

In [4]:
import os, re, warnings
import numpy as np
import pandas as pd
import mne
from scipy.sparse import csr_matrix

# Condiciones a recorrer
conditions = ["zinnen", "woorden"]

# Diccionarios para guardar info
all_ch_sets = {}
all_ch_names = {}
all_adjacencies = {}

# Diccionarios para guardar info
all_ch_sets = {}
all_ch_names = {}
all_adjacencies = {}

for subj in subj_epochs_block:
    
    # Archivo único con todas las condiciones
    fname = epochs_clean_path / f"{subj}_epochs_{layer_script}-epo.fif"
    
    if not fname.exists():
        warnings.warn(f"⚠️ Archivo no encontrado: {fname}")
        continue

    # Cargar epochs del sujeto
    epochs = mne.read_epochs(fname)

    # Seleccionar solo magnetómetros y quitar bads
    epochs_mag_clean = epochs.copy().pick(picks="mag", exclude='bads')

    # Canales realmente presentes y válidos
    canales_efectivos = set(epochs_mag_clean.ch_names)

    # Obtener adyacencia de todos los magnetómetros del sensor layout
    adjacency, ch_names_total = mne.channels.find_ch_adjacency(
        epochs.info, ch_type='mag'
    )
    ch_names_total_str = [str(ch) for ch in ch_names_total]

    # Guardar info usando solo la clave del sujeto
    all_ch_sets[subj] = canales_efectivos
    all_ch_names[subj] = ch_names_total_str
    all_adjacencies[subj] = adjacency

    del epochs, epochs_mag_clean
# --- Comprobación de consistencia ---
ref_key = next(iter(all_ch_names.keys()))
ref_channels = set(all_ch_names[ref_key])
ref_adjacency = all_adjacencies[ref_key]

warnings_flag = False
for key, chs in all_ch_names.items():
    diff1 = set(chs) - ref_channels
    diff2 = ref_channels - set(chs)
    if diff1 or diff2:
        warnings.warn(f"⚠️ Diferencia de canales en {key}: {diff1} {diff2}")
        warnings_flag = True

if warnings_flag:
    print("❌ No se puede crear adjacency_reducida porque hay diferencias de canales.")
    adjacency_reducida = None
else:
    print("✅ Todos los sujetos/condiciones tienen los mismos canales.")
    
    # Usamos el set de canales efectivos de referencia
    canales_efectivos = all_ch_sets[ref_key]

    # Creamos tabla de referencia
    df_canales = pd.DataFrame({
        'indice': range(len(ref_channels)),
        'nombre_base': list(ref_channels),
        'nombre_con_sufijo': [f"{ch}-4304" for ch in ref_channels]
    })

    df_canales[f"canal_efectivo"] = df_canales['nombre_con_sufijo'].apply(
        lambda ch: ch if ch in canales_efectivos else np.nan
    )

    indices_efectivos = df_canales.loc[
        df_canales['canal_efectivo'].notna(), 'indice'
    ].to_list()

    adjacency_reducida = ref_adjacency[indices_efectivos, :][:, indices_efectivos]
    print(f"✅ adjacency_reducida global creada con forma: {adjacency_reducida.shape}")

Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1100_epochs_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
39 matching events found
No baseline correction applied
0 projection items activated
Reading adjacency matrix for ctf275.
Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1057_epochs_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
46 matching events found
No baseline correction applied
0 projection items activated
Reading adjacency matrix for ctf275.
Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1011_epochs_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting 

KeyboardInterrupt: 

In [ ]:
# Nombre completo del archivo
csv_path = os.path.join(channels_structure_path, f"channels_mag_{modality}.csv")

# Guardar el DataFrame
df_canales.to_csv(csv_path, index=False)

print(f"✅ Archivo guardado como: {csv_path}")
####lectura de la matriz de adyacencia (tiene que ir despue por fuerza)

import pickle
with open(channels_structure_path / f"adjacency_reduced_{modality}.pkl", "wb") as f:
    pickle.dump(adjacency_reducida, f)


✅ Archivo guardado como: g:\MOUS_204\channels_structure\channels_mag_visual.csv


# original

In [ ]:

epochs_woorden=mne.read_epochs(epochs_clean_path / f"{subj}_epochs_woorden_{layer_script}-epo.fif")

In [ ]:

#se coge este sujeto, entiendo porque tiene los 272 canales comunes a todos los sujetos
epochs_mag_clean = epochs_woorden.copy().pick(picks="mag", exclude='bads')

print(len(epochs_mag_clean.ch_names))  # Esto debería darte 272

# Canales efectivos reales del objeto epochs (ya vienen con -4304)
canales_efectivos = set(epochs_mag_clean.ch_names)  # conjunto para acceso rápido



# Obtenemos matriz de adjacencia general, y el total de los canales
adjacency, ch_names_total = mne.channels.find_ch_adjacency(epochs_woorden.info, ch_type='mag')
print(adjacency.shape)  # Esto debería darte (272, 272)

# Convertimos a string normal por si acaso (np.str_ a str)
ch_names_total_str = [str(ch) for ch in ch_names_total]

# Creamos la tabla de los canales con sus nombres y sufijos
df_canales = pd.DataFrame({
    'indice': range(len(ch_names_total_str)),
    'nombre_base': ch_names_total_str,
    'nombre_con_sufijo': [f"{ch}-4304" for ch in ch_names_total_str]
})

# Mostramos las primeras filas
print(df_canales.head())


# Añadimos la columna 'canal_efectivo': si está en el set, lo ponemos, si no, NaN
df_canales[f"canal_efectivo_{modality}"] = df_canales['nombre_con_sufijo'].apply(
    lambda ch: ch if ch in canales_efectivos else np.nan
)

indices_efectivos = df_canales.loc[df_canales[f'canal_efectivo_{modality}'].notna(), 'indice'].to_list()

# Paso 2: Recortar la matriz de adyacencia original
from scipy.sparse import csr_matrix

# adjacency_total es sparse, así que podemos hacer slicing con arrays de índices
adjacency_reducida = adjacency[indices_efectivos, :][:, indices_efectivos]

# Confirmamos la forma
print(f"✅ adjacency_reducida creada con forma: {adjacency_reducida.shape}")

In [ ]:
# Nombre completo del archivo
csv_path = os.path.join(channels_structure_path, f"channels_mag_{modality}.csv")

# Guardar el DataFrame
df_canales.to_csv(csv_path, index=False)

print(f"✅ Archivo guardado como: {csv_path}")

In [ ]:
####lectura de la matriz de adyacencia (tiene que ir despue por fuerza)

import pickle
with open(channels_structure_path / f"adjacency_reduced_{modality}.pkl", "wb") as f:
    pickle.dump(adjacency_reducida, f)
